# Load Engineered Features
Pull the finalized, multi-dimensional feature dataset straight into the modeling workflow.

In [0]:
feature_df = spark.table("workspace.default.backblaze_features_delta")

# Temporal Train/Test Split (Prevent Data Leakage)
A random split will ruin a time-series project because data from the same hard drive would bleed into both sets. We will enforce a strict chronological boundary: use October and November for training, and hold out December completely to act as our live deployment simulator.

In [0]:
train_df = feature_df.filter(feature_df.date < '2025-12-01')
test_df = feature_df.filter(feature_df.date >= '2025-12-01')

print(f"Train rows: {train_df.count()}")
print(f"Test rows: {test_df.count()}")

Train rows: 20453101
Test rows: 10488607


# Compute and Append Dynamic Class Weights
Because hard drive crashes are incredibly rare, standard models will cheat and predict "Healthy" every single time to artificially inflate their accuracy score. We need to calculate a penalty scaling factor that forces the model to treat missing a single failure as a catastrophic mathematical error.
$$\text{Weight Factor} = \frac{\text{Total Count of Healthy Rows (0)}}{\text{Total Count of Failing Rows (1)}}$$

In [0]:
from pyspark.sql import functions as F

count_0 = train_df.filter(train_df.label == 0).count()
count_1 = train_df.filter(train_df.label == 1).count()
weight_factor = count_0 / count_1

weighted_train_df = train_df.withColumn(
    "weight",
    F.when(F.col("label") == 1, F.lit(weight_factor)).otherwise(F.lit(1.0))
)

# Construct the Spark ML Pipeline
Now, wire up the end-to-end extraction and classification pipeline. We will package it neatly inside pyspark.ml.Pipeline so it can be deployed or scaled seamlessly.

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml import Pipeline

assembler = VectorAssembler(
    inputCols=[
        'smart_5_raw',
        'smart_187_raw',
        'smart_197_raw',
        'smart_198_raw',
        'smart_5_delta_7',
        'smart_187_delta_7'
    ],
    outputCol='features'
)

gbt = GBTClassifier(
    labelCol='label',
    featuresCol='features',
    weightCol='weight'
)

pipeline = Pipeline(stages=[assembler, gbt])

final_gbt_model = pipeline.fit(weighted_train_df)

# Evaluate on the Test Set (December Data Simulator)
Now, pass the unseen December test data through the trained model structure. We will bypass basic accuracy and look directly at AUC-ROC and a raw Confusion Matrix to track exact Precision and Recall counts.

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F

predictions = final_gbt_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator.evaluate(predictions)
print(f"AUC-ROC: {auc}")

predictions = predictions.withColumn(
    "prediction_label",
    F.when(F.col("prediction") == 1.0, F.lit(1)).otherwise(F.lit(0))
)

confusion_matrix = predictions.crosstab('label', 'prediction_label')
display(confusion_matrix)

AUC-ROC: 0.8603523004728927


label_prediction_label,0,1
0,10028402,458638
1,400,1167


In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F

# Generate predictions
predictions = final_gbt_model.transform(test_df)

# Calculate AUC-ROC
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator.evaluate(predictions)
print(f"AUC-ROC: {auc}")

# Compute confusion matrix
confusion_matrix = predictions.withColumn(
    "prediction_label",
    F.when(F.col("prediction") == 1.0, 1).otherwise(0)
).groupBy("label", "prediction_label").count()

# Pivot to get TP, FP, TN, FN
confusion_counts = confusion_matrix.groupBy().pivot(
    F.concat(F.lit("label_"), F.col("label"), F.lit("_pred_"), F.col("prediction_label"))
).agg(F.first("count"))

display(confusion_matrix)

AUC-ROC: 0.8603523004728927


---------------------------------------------------------------------------
PySparkTypeError Traceback (most recent call last)
File , line 23
 17 confusion_matrix = predictions.withColumn(
 18 "prediction_label",
 19 F.when(F.col("prediction") == 1.0, 1).otherwise(0)
 20 ).groupBy("label", "prediction_label").count()
 22 # Pivot to get TP, FP, TN, FN
---> 23 confusion_counts = confusion_matrix.groupBy().pivot(
 24 F.concat(F.lit("label_"), F.col("label"), F.lit("_pred_"), F.col("prediction_label"))
 25 ).agg(F.first("count"))
 27 display(confusion_matrix)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/group.py:237, in GroupedData.pivot(self, pivot_col, values)
 231 raise PySparkNotImplementedError(
 232 errorClass="UNSUPPORTED_OPERATION",
 233 messageParameters={"operation": f"PIVOT after {self._group_type.upper()}"},
 234 )
 236 if not isinstance(pivot_col, str):
--> 237 raise PySparkTypeError(
 238 errorClass="NOT_STR",
 239 messageParameters={"arg_name": "pivot_col", "arg_type": type(pivot_col).__name__},
 240 )
 242 if values is not None:
 243 if not isinstance(values, list):

PySparkTypeError: [NOT_STR] Argument `pivot_col` should be a str, got Column.

The `crosstab` function outputs the raw counts of a **Confusion Matrix**, meaning Precision and Recall aren't printed directly—you have to calculate them using the four quadrants of the table.

Because you used `.crosstab("label", "prediction_label")`, the **rows** represent the actual reality, and the **columns** represent what your model guessed.

---

### Mapping Your Confusion Matrix

Here is how your specific numbers map out to standard machine learning evaluation metrics:

* **True Negatives (TN):** **`10,028,402`** (Drives that were healthy, and the model correctly predicted they were healthy)
* **False Positives (FP):** **`458,638`** (Drives that were healthy, but the model flagged them as failing)
* **False Negatives (FN):** **`400`** (Drives that actually failed, but your model missed them)
* **True Positives (TP):** **`1,167`** (Drives that actually failed, and your model correctly caught them)

---

### Step-by-Step Calculations

#### 1. Recall (Sensitivity)
Recall answers: "Out of all the hard drives that actually died in December, what percentage did my model successfully catch ahead of time?"

$$Recall = \frac{True\ Positives}{True\ Positives + False\ Negatives}\%$$

$$Recall = \frac{1,167}{1,167 + 400} = \frac{1,167}{1,567} = 74.47\%$$
#### 2. Precision
Precision answers: "When my model flags a drive as failing, how often is it actually right?"
$$Precision = \frac{True\ Positives}{True\ Positives + False\ Positives}$$

$$Precision = \frac{1,167}{1,167 + 458,638} = \frac{1,167}{459,805} = 0.25\%$$

---